# 📊 Project Lessons Learned — Text Analytics Notebook

A self-contained notebook that reads any Excel file with text-only columns and produces
**interactive inline visualisations** across every analysis technique below.
At the end, it also generates a polished **Excel workbook** and **PowerPoint presentation**.

---

| # | Section | Technique |
|---|---------|-----------|
| 1 | Load & Explore | DataFrame preview, column mapping |
| 2 | Distribution Analysis | Bar charts by Type, Stage, Category, Org |
| 3 | Threat vs. Opportunity | Pie split + keyword comparison |
| 4 | Keyword Frequency | Top-N word frequency per text field |
| 5 | Word Cloud | Visual theme map |
| 6 | Stage × Category | Stacked bar + heatmap |
| 7 | N-gram Analysis | Top bigrams & trigrams |
| 8 | Sentiment Analysis | VADER scores, distribution charts |
| 9 | Text Length & Readability | Flesch-Kincaid grade level |
| 10 | TF-IDF by Category | Distinctive (not just frequent) keywords |
| 11 | LDA Topic Modeling | Latent topic discovery |
| 12 | Document Clustering | TF-IDF + K-Means + PCA scatter |
| 13 | Named Entity Recognition | People, orgs, locations |
| 14 | Co-occurrence Network | Word relationship graph |
| 15 | Sankey Flow | Type → Stage → Category flow |
| 16 | Semantic Similarity | Sentence embedding heatmap |
| 📤 | Export | Excel workbook + PowerPoint deck |

> **All sections beyond Section 2 require optional packages — see the Setup cell.**

## 🔧 Setup

Run the cell below once to install everything. Comment out lines you don't need.

```bash
pip install pandas openpyxl python-pptx matplotlib          # required
pip install nltk wordcloud pillow                            # word cloud + VADER sentiment
pip install scikit-learn                                     # n-grams, TF-IDF, LDA, clustering
pip install textstat                                         # readability metrics
pip install networkx                                         # co-occurrence network
pip install spacy && python -m spacy download en_core_web_sm # NER
pip install plotly kaleido                                   # Sankey diagram
pip install sentence-transformers                            # semantic similarity
```

In [ ]:
import sys, re, warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

%matplotlib inline
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#FAFAFA",
    "axes.facecolor":   "#FAFAFA",
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})

# ── Optional packages ──────────────────────────────────────────────────────────
try:
    import nltk
    from nltk.corpus import stopwords as nltk_sw
    from nltk.stem import WordNetLemmatizer
    for _p in ["punkt","stopwords","wordnet","omw-1.4","punkt_tab","vader_lexicon"]:
        try: nltk.download(_p, quiet=True)
        except: pass
    NLTK_OK = True
except ImportError:
    NLTK_OK = False

try:
    from wordcloud import WordCloud; WC_OK = True
except ImportError:
    WC_OK = False

try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA, LatentDirichletAllocation
    from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False

try:
    import textstat; TEXTSTAT_OK = True
except ImportError:
    TEXTSTAT_OK = False

try:
    import networkx as nx; NX_OK = True
except ImportError:
    NX_OK = False

try:
    import spacy; _nlp = spacy.load("en_core_web_sm"); SPACY_OK = True
except (ImportError, OSError):
    SPACY_OK = False; _nlp = None

try:
    import plotly.graph_objects as go; PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

try:
    from sentence_transformers import SentenceTransformer; SBERT_OK = True
except ImportError:
    SBERT_OK = False

status = {
    "nltk": NLTK_OK, "wordcloud": WC_OK, "sklearn": SKLEARN_OK,
    "textstat": TEXTSTAT_OK, "networkx": NX_OK,
    "spacy": SPACY_OK, "plotly": PLOTLY_OK, "sentence-transformers": SBERT_OK,
}
print("Package status:")
for pkg, ok in status.items():
    print(f"  {'✅' if ok else '❌'} {pkg}")

In [ ]:
# ── Design palette ────────────────────────────────────────────────────────────
PAL = {
    "primary": "1C4E80", "secondary": "0091D5", "accent1": "EA6A47",
    "accent2": "A5D8DD", "accent3": "488A99", "light": "F0F4F8",
    "dark": "1A2B3C", "threat": "C0392B", "opportunity": "27AE60",
}
C = {k: f"#{v}" for k, v in PAL.items()}   # shorthand hex colours
MULTI = [C["primary"], C["secondary"], C["accent1"], C["accent2"], C["accent3"],
         "#7EC8E3", "#FFD166", "#EF476F", "#06D6A0", "#118AB2"] * 4

# ── Column aliases ─────────────────────────────────────────────────────────────
ALIASES = {
    "project_name":      ["project name", "project"],
    "organization":      ["organization", "org", "department"],
    "project_type":      ["project type", "type"],
    "project_stage":     ["project stage", "stage", "phase"],
    "original_category": ["original category", "orig category", "orig cat"],
    "new_category":      ["new category", "revised category", "new cat"],
    "lesson_title":      ["lesson learned title", "lesson title", "title"],
    "threat_opp_type":   ["threat/opportunity", "threat or opportunity", "risk type"],
    "description":       ["description of threat/opportunity", "description", "desc"],
    "response":          ["response of threat/opportunity", "response", "mitigation"],
    "lessons_learned":   ["lessons learned", "lesson learned", "lessons", "key takeaway"],
}
EXTRA_SW = {
    "project","use","used","using","need","ensure","must","also","may","well",
    "one","two","make","team","time","new","work","include","process","provide",
    "within","would","could","should","will","many","often","however","therefore",
    "due","per","key","based","required","high","low","good","important",
}

print("Constants loaded.")

In [ ]:
# ── Utility functions ──────────────────────────────────────────────────────────

def map_columns(df):
    lower = {c.lower().strip(): c for c in df.columns}
    return {canon: lower[a] for canon, aliases in ALIASES.items()
            for a in aliases if a in lower}

def build_stopwords():
    sw = EXTRA_SW.copy()
    if NLTK_OK:
        sw |= set(nltk_sw.words("english"))
    else:
        sw |= {"the","a","an","and","or","but","in","on","at","to","for","of",
               "with","by","from","is","are","was","were","be","been","have",
               "has","had","do","does","did","not","no","it","its","they",
               "them","their","we","our","you","your","this","that","these","those"}
    return sw

def tokenize(text, sw):
    tokens = re.sub(r"[^a-z\s]", " ", str(text).lower()).split()
    if NLTK_OK:
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(t) for t in tokens]
    return [t for t in tokens if t not in sw and len(t) > 2]

def top_keywords(texts, sw, n=20):
    c = Counter()
    for t in texts: c.update(tokenize(t, sw))
    return c.most_common(n)

def freq_table(df, col, label="Value"):
    counts = df[col].value_counts().reset_index()
    counts.columns = [label, "Count"]
    counts["Pct"] = (counts["Count"] / counts["Count"].sum() * 100).round(1)
    return counts

def split_threats_opps(df, col_map):
    dc, tc = col_map.get("description",""), col_map.get("threat_opp_type","")
    if not dc: return [], []
    threats, opps = [], []
    for _, row in df.iterrows():
        txt = str(row.get(dc, ""))
        lbl = str(row.get(tc, "")).lower() if tc else ""
        (opps if "opportunit" in lbl else threats).append(txt)
    return threats, opps

def hbar(ax, labels, values, color, title):
    """Simple horizontal bar helper."""
    pos = range(len(labels))
    ax.barh(pos, values, color=color, edgecolor="white", linewidth=0.5)
    ax.set_yticks(pos); ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    for i, v in enumerate(values):
        ax.text(v + max(values)*0.01, i, str(v), va="center", fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold", color=C["dark"], pad=8)
    ax.set_xlabel("Count", fontsize=9)

def vbar(ax, labels, values, color, title):
    """Simple vertical bar helper."""
    pos = range(len(labels))
    ax.bar(pos, values, color=color, edgecolor="white", linewidth=0.5)
    ax.set_xticks(pos)
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
    for i, v in enumerate(values): ax.text(i, v+max(values)*0.01, str(v), ha="center", fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold", color=C["dark"], pad=8)
    ax.set_ylabel("Count", fontsize=9)

print("Utility functions ready.")

---
## ⚙️ Configuration — Start Here

Set `INPUT_FILE` to your Excel file path. Everything else is optional.

In [ ]:
INPUT_FILE = "your_data.xlsx"   # ← change this to your file path
SHEET_NAME = None               # None = first sheet, or e.g. "Sheet2"
OUT_DIR    = "."                # output directory for Excel + PowerPoint export
TOP_N      = 15                 # top-N items in most charts

---
## 1️⃣ Load & Explore Data

Reads the Excel file, auto-maps column names, and builds a stopword set.

In [ ]:
xl = pd.ExcelFile(INPUT_FILE)
df = pd.read_excel(xl, sheet_name=SHEET_NAME or xl.sheet_names[0], dtype=str).fillna("")
df.columns = df.columns.str.strip()

col_map = map_columns(df)
sw      = build_stopwords()

print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print(f"\nRecognised fields ({len(col_map)}):")
for canon, actual in col_map.items():
    print(f"  {canon:25s} → '{actual}'")

In [ ]:
display(df.head(5).style.set_properties(**{"font-size": "11px"}))

---
## 2️⃣ Distribution Analysis

How are records spread across Project Type, Stage, Category, and Organization?

In [ ]:
fields = [
    ("project_type",  C["primary"],    "Project Type"),
    ("project_stage", C["secondary"],  "Project Stage"),
    ("new_category",  C["accent1"],    "Category"),
    ("organization",  C["accent3"],    "Organization"),
]
active = [(f, color, label) for f, color, label in fields if col_map.get(f)]

if not active:
    print("No recognised distribution columns found.")
else:
    fig, axes = plt.subplots(1, len(active), figsize=(5.5 * len(active), 4.5))
    if len(active) == 1: axes = [axes]
    for ax, (field, color, label) in zip(axes, active):
        col   = col_map[field]
        df_f  = freq_table(df, col, label)
        lbls  = df_f[label].tolist()[:12]
        vals  = df_f["Count"].tolist()[:12]
        use_h = max(len(str(l)) for l in lbls) > 10
        if use_h:
            hbar(ax, lbls, vals, color, f"By {label}")
        else:
            vbar(ax, lbls, vals, color, f"By {label}")
    plt.suptitle("Record Distribution", fontsize=14, fontweight="bold",
                 color=C["dark"], y=1.02)
    plt.tight_layout()
    plt.show()

# Print frequency tables
for field, _, label in active:
    col = col_map[field]
    print(f"\n{label}:")
    display(freq_table(df, col, label).head(10))

---
## 3️⃣ Threat vs. Opportunity

Compares the volume and top keywords between threat and opportunity records.

In [ ]:
to_col = col_map.get("threat_opp_type")
t_texts, o_texts = split_threats_opps(df, col_map)

if not to_col and not (t_texts or o_texts):
    print("No Threat/Opportunity column found.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Pie chart
    if to_col:
        df_to = freq_table(df, to_col, "Type")
        axes[0].pie(df_to["Count"], labels=df_to["Type"],
                    colors=MULTI[:len(df_to)], autopct="%1.1f%%",
                    startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5))
        axes[0].set_title("Threat vs. Opportunity Split", fontsize=11,
                           fontweight="bold", color=C["dark"])
    else:
        axes[0].axis("off")
        axes[0].text(0.5, 0.5, "No type column", ha="center", va="center",
                     transform=axes[0].transAxes)

    # Threat keywords
    t_kws = top_keywords(t_texts, sw, TOP_N) if t_texts else []
    if t_kws:
        hbar(axes[1], [k for k,_ in t_kws], [v for _,v in t_kws],
             C["threat"], f"Top {TOP_N} Threat Keywords")
    else:
        axes[1].axis("off")

    # Opportunity keywords
    o_kws = top_keywords(o_texts, sw, TOP_N) if o_texts else []
    if o_kws:
        hbar(axes[2], [k for k,_ in o_kws], [v for _,v in o_kws],
             C["opportunity"], f"Top {TOP_N} Opportunity Keywords")
    else:
        axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"Threats: {len(t_texts)}  |  Opportunities: {len(o_texts)}")

---
## 4️⃣ Keyword Frequency

Top words extracted from each text field after removing stop words and lemmatizing.

In [ ]:
kw_fields = [
    ("description",    C["primary"],   "Description"),
    ("response",       C["accent3"],   "Response / Mitigation"),
    ("lessons_learned",C["secondary"], "Lessons Learned"),
    ("lesson_title",   C["accent1"],   "Lesson Titles"),
]
kw_active = [(f, color, label) for f, color, label in kw_fields if col_map.get(f)]

for field, color, label in kw_active:
    col  = col_map[field]
    kws  = top_keywords(df[col].tolist(), sw, TOP_N)
    if not kws: continue
    fig, ax = plt.subplots(figsize=(9, 4.5))
    hbar(ax, [k for k,_ in kws], [v for _,v in kws], color,
         f"Top {TOP_N} Keywords — {label}")
    plt.tight_layout(); plt.show()

---
## 5️⃣ Word Cloud

Visual frequency map of all combined text. Requires `wordcloud` package.

In [ ]:
if not WC_OK:
    print("Install wordcloud:  pip install wordcloud pillow")
else:
    all_text_cols = [col_map.get(f) for f in
                     ["description","response","lessons_learned","lesson_title"]
                     if col_map.get(f)]
    all_texts = [t for col in all_text_cols for t in df[col].tolist()]
    combined  = " ".join(" ".join(tokenize(t, sw)) for t in all_texts)

    if combined.strip():
        wc = WordCloud(width=1000, height=480, background_color="white",
                       colormap="Blues", max_words=80, stopwords=sw,
                       collocations=False).generate(combined)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title("Word Cloud — All Text Fields", fontsize=14,
                     fontweight="bold", color=C["dark"], pad=10)
        plt.tight_layout(); plt.show()
    else:
        print("Not enough text to generate a word cloud.")

---
## 6️⃣ Stage × Category Matrix

Stacked bar chart and a colour-coded pivot table showing how lesson categories distribute across project stages.

In [ ]:
sc = col_map.get("project_stage")
cc = col_map.get("new_category") or col_map.get("original_category")

if not sc or not cc:
    print("Need both project_stage and a category column for this section.")
else:
    pivot = pd.crosstab(df[sc], df[cc])

    # Stacked bar
    fig, ax = plt.subplots(figsize=(12, 5))
    bottom = np.zeros(len(pivot))
    for i, col in enumerate(pivot.columns):
        ax.bar(pivot.index, pivot[col], bottom=bottom,
               label=col, color=MULTI[i % len(MULTI)],
               edgecolor="white", linewidth=0.5)
        bottom += pivot[col].values
    ax.set_title("Lessons by Stage & Category", fontsize=12,
                 fontweight="bold", color=C["dark"], pad=8)
    ax.set_ylabel("Count", fontsize=9)
    ax.set_xticklabels(pivot.index, rotation=30, ha="right", fontsize=9)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()

    # Heatmap table
    max_val = pivot.values.max() if pivot.values.max() > 0 else 1
    styled  = pivot.style.background_gradient(cmap="Blues", axis=None)
    display(styled)

---
## 7️⃣ N-gram Analysis

Bigrams (2-word phrases) and trigrams (3-word phrases) reveal context that single-word frequency misses.
Requires `scikit-learn`.

In [ ]:
if not SKLEARN_OK:
    print("Install scikit-learn:  pip install scikit-learn")
else:
    ng_fields = [("lessons_learned", C["secondary"], "Lessons Learned"),
                 ("description",     C["primary"],   "Description")]
    ng_colors = [C["secondary"], C["primary"], C["accent1"]]

    for field, color, label in ng_fields:
        col = col_map.get(field)
        if not col: continue
        texts = df[col].tolist()

        fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
        for ax, (n, name) in zip(axes, [(2, "Bigrams"), (3, "Trigrams")]):
            try:
                vec = CountVectorizer(ngram_range=(n,n), stop_words=list(sw),
                                      max_features=500, min_df=1)
                X   = vec.fit_transform(texts)
                cnt = X.sum(axis=0).A1
                nms = vec.get_feature_names_out()
                top = sorted(zip(nms, cnt), key=lambda x: -x[1])[:TOP_N]
                if top:
                    hbar(ax, [p for p,_ in top], [int(c) for _,c in top],
                         color, f"Top {TOP_N} {name} — {label}")
                else:
                    ax.axis("off")
            except Exception as e:
                ax.axis("off"); ax.text(0.5, 0.5, str(e), ha="center", transform=ax.transAxes)
        plt.tight_layout(); plt.show()

---
## 8️⃣ Sentiment Analysis (VADER)

VADER assigns each record a **compound score** from −1 (very negative) to +1 (very positive).
Requires `nltk` (already listed as optional above).

In [ ]:
lc = col_map.get("lessons_learned") or col_map.get("description")

if not NLTK_OK or not lc:
    print("Requires nltk and a lessons_learned / description column.")
else:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sia    = SentimentIntensityAnalyzer()
    scores = df[lc].apply(lambda t: sia.polarity_scores(str(t)))
    sent   = pd.DataFrame(scores.tolist())
    sent["label"] = sent["compound"].apply(
        lambda s: "Positive" if s >= 0.05 else ("Negative" if s <= -0.05 else "Neutral"))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Pie
    lbl_cnt = sent["label"].value_counts()
    clr_map = {"Positive": C["opportunity"], "Neutral": C["secondary"], "Negative": C["threat"]}
    clrs    = [clr_map.get(l, C["accent2"]) for l in lbl_cnt.index]
    axes[0].pie(lbl_cnt.values, colors=clrs, autopct="%1.1f%%", startangle=140,
                wedgeprops=dict(edgecolor="white", linewidth=1.5))
    axes[0].legend(lbl_cnt.index, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)
    axes[0].set_title("Sentiment Distribution", fontsize=11,
                      fontweight="bold", color=C["dark"])

    # Histogram
    axes[1].hist(sent["compound"], bins=25, color=C["secondary"],
                 edgecolor="white", linewidth=0.5)
    axes[1].axvline( 0.05, color=C["opportunity"], linestyle="--",
                    linewidth=1.5, label="Positive (+0.05)")
    axes[1].axvline(-0.05, color=C["threat"],      linestyle="--",
                    linewidth=1.5, label="Negative (−0.05)")
    axes[1].set_xlabel("Compound Score", fontsize=9)
    axes[1].set_ylabel("Count", fontsize=9)
    axes[1].set_title("Score Distribution (VADER Compound)", fontsize=11,
                      fontweight="bold", color=C["dark"])
    axes[1].legend(fontsize=9)

    plt.tight_layout(); plt.show()

    print(f"\nSentiment summary:")
    display(sent["label"].value_counts().rename("Count").reset_index())
    print(f"\nMean compound score: {sent['compound'].mean():.3f}")

    # Attach to main df for use in later sections
    df["_sentiment_label"]    = sent["label"].values
    df["_sentiment_compound"] = sent["compound"].values

---
## 9️⃣ Text Length & Readability

**Length** catches data quality issues (very short entries may be incomplete).
**Flesch-Kincaid grade level** measures writing complexity — lower = easier to read.
Requires `textstat`.

In [ ]:
lc = col_map.get("lessons_learned") or col_map.get("description")

if not lc:
    print("No lessons_learned / description column found.")
else:
    lengths = df[lc].apply(lambda t: len(str(t).split()))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Length histogram
    axes[0].hist(lengths, bins=min(30, max(len(lengths)//3, 5)),
                 color=C["accent2"], edgecolor="white", linewidth=0.5)
    axes[0].axvline(lengths.mean(), color=C["accent1"], linestyle="--",
                    linewidth=1.5, label=f"Mean: {lengths.mean():.1f} words")
    axes[0].set_xlabel("Word Count per Record", fontsize=9)
    axes[0].set_ylabel("Frequency", fontsize=9)
    axes[0].set_title("Text Length Distribution", fontsize=11,
                      fontweight="bold", color=C["dark"])
    axes[0].legend(fontsize=9)

    # Readability bar
    if not TEXTSTAT_OK:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "pip install textstat\nfor readability",
                     ha="center", va="center", transform=axes[1].transAxes, fontsize=11)
    else:
        group_col = (col_map.get("project_stage") or
                     col_map.get("new_category") or
                     col_map.get("original_category"))
        if group_col:
            def fk(t):
                words = str(t).split()
                return textstat.flesch_kincaid_grade(str(t)) if len(words) >= 5 else None
            fk_scores = df[lc].apply(fk)
            avg = (pd.concat([df[[group_col]], fk_scores.rename("fk")], axis=1)
                   .groupby(group_col)["fk"].mean().dropna().sort_values())
            if len(avg) >= 2:
                hbar(axes[1], avg.index.tolist(),
                     avg.round(1).tolist(), C["accent1"],
                     f"Avg F-K Grade by {group_col.replace('_',' ').title()}")
            else:
                axes[1].axis("off")
        else:
            axes[1].axis("off")

    plt.tight_layout(); plt.show()
    print(f"Word count — min: {lengths.min()}, max: {lengths.max()}, "
          f"mean: {lengths.mean():.1f}, median: {lengths.median():.1f}")

---
## 🔟 TF-IDF Keywords by Category

TF-IDF finds the words that are **most distinctive** to each category — not just most frequent overall.
Words common everywhere score low; words unique to one category score high.
Requires `scikit-learn`.

In [ ]:
cat_col  = col_map.get("new_category") or col_map.get("original_category")
text_col = col_map.get("lessons_learned") or col_map.get("description")

if not SKLEARN_OK or not cat_col or not text_col:
    print("Requires scikit-learn, a category column, and a text column.")
else:
    categories = df[cat_col].unique()
    results    = {}
    for cat in categories:
        texts = df[df[cat_col] == cat][text_col].tolist()
        if len(texts) < 2: continue
        try:
            vec = TfidfVectorizer(stop_words=list(sw), max_features=300)
            X   = vec.fit_transform(texts)
            scores = X.mean(axis=0).A1
            names  = vec.get_feature_names_out()
            top    = sorted(zip(names, scores), key=lambda x: -x[1])[:10]
            results[str(cat)] = top
        except Exception:
            continue

    if results:
        n_cats = len(results)
        fig, axes = plt.subplots(1, n_cats, figsize=(4.5 * n_cats, 5))
        if n_cats == 1: axes = [axes]
        for ax, (cat, top) in zip(axes, results.items()):
            lbls = [w for w,_ in top]
            vals = [round(s, 3) for _,s in top]
            ax.barh(range(len(lbls)), vals, color=C["accent3"],
                    edgecolor="white", linewidth=0.5)
            ax.set_yticks(range(len(lbls))); ax.set_yticklabels(lbls, fontsize=8)
            ax.invert_yaxis()
            ax.set_title(str(cat)[:25], fontsize=9, fontweight="bold", color=C["dark"])
            ax.set_xlabel("Avg TF-IDF", fontsize=8)
        plt.suptitle("TF-IDF Distinctive Keywords per Category", fontsize=12,
                     fontweight="bold", color=C["dark"])
        plt.tight_layout(); plt.show()

        # Table view
        rows = []
        for cat, top in results.items():
            for rank, (word, score) in enumerate(top, 1):
                rows.append({"Category": cat, "Rank": rank, "Keyword": word,
                             "TF-IDF Score": round(score, 4)})
        display(pd.DataFrame(rows).head(40))

---
## 1️⃣1️⃣ LDA Topic Modeling

Latent Dirichlet Allocation discovers hidden thematic clusters without needing pre-defined labels.
Requires `scikit-learn`. Works best with 50+ records.

In [ ]:
lc = col_map.get("lessons_learned") or col_map.get("description")

if not SKLEARN_OK or not lc or len(df) < 10:
    print("Requires scikit-learn, a text column, and ≥10 records.")
else:
    lesson_texts = df[lc].tolist()
    n_topics = min(5, max(2, len(lesson_texts) // 10))

    try:
        vec = CountVectorizer(stop_words=list(sw), max_features=300, min_df=2)
        X   = vec.fit_transform(lesson_texts)
        lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=30)
        lda.fit(X)
        feat_names = vec.get_feature_names_out()
        topics = [[feat_names[i] for i in comp.argsort()[:-11:-1]]
                  for comp in lda.components_]

        fig, axes = plt.subplots(1, n_topics, figsize=(3.8 * n_topics, 5))
        if n_topics == 1: axes = [axes]
        for i, (ax, words) in enumerate(zip(axes, topics)):
            scores = list(range(len(words), 0, -1))
            ax.barh(range(len(words)), scores,
                    color=MULTI[i % len(MULTI)], edgecolor="white")
            ax.set_yticks(range(len(words))); ax.set_yticklabels(words, fontsize=8)
            ax.set_title(f"Topic {i+1}", fontweight="bold", fontsize=10, color=C["dark"])
            ax.set_xticks([])
            for spine in ax.spines.values(): spine.set_visible(False)
        plt.suptitle(f"LDA Topic Modeling — {n_topics} Topics Discovered",
                     fontsize=12, fontweight="bold", color=C["dark"])
        plt.tight_layout(); plt.show()

        print("\nTop 10 terms per topic:")
        for i, words in enumerate(topics):
            print(f"  Topic {i+1}: {', '.join(words)}")

    except Exception as e:
        print(f"LDA failed: {e}")

---
## 1️⃣2️⃣ Document Clustering (PCA)

Groups similar records using TF-IDF vectorization and K-Means, then projects to 2D with PCA.
Each dot = one record; colour = cluster. Shows natural groupings beyond predefined categories.
Requires `scikit-learn`.

In [ ]:
lc = col_map.get("lessons_learned") or col_map.get("description")

if not SKLEARN_OK or not lc or len(df) < 12:
    print("Requires scikit-learn, a text column, and ≥12 records.")
else:
    n_clusters = min(5, max(2, len(df) // 15))
    texts = df[lc].tolist()

    try:
        vec    = TfidfVectorizer(max_features=500, min_df=1)
        X      = vec.fit_transform(texts)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        pca    = PCA(n_components=2, random_state=42)
        coords = pca.fit_transform(X.toarray())

        fig, ax = plt.subplots(figsize=(9, 6))
        for ci in range(n_clusters):
            mask = labels == ci
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=MULTI[ci], label=f"Cluster {ci+1}",
                       alpha=0.75, s=60, edgecolors="white", linewidth=0.5)
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)", fontsize=9)
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)", fontsize=9)
        ax.set_title(f"Document Clusters — TF-IDF + K-Means + PCA  ({n_clusters} clusters)",
                     fontsize=11, fontweight="bold", color=C["dark"], pad=8)
        ax.legend(fontsize=9)
        plt.tight_layout(); plt.show()

        # Cluster size summary
        cluster_sizes = pd.Series(labels).value_counts().sort_index()
        cluster_sizes.index = [f"Cluster {i+1}" for i in cluster_sizes.index]
        display(cluster_sizes.rename("Records").reset_index())

    except Exception as e:
        print(f"Clustering failed: {e}")

---
## 1️⃣3️⃣ Named Entity Recognition (NER)

Extracts real-world entities — people, organizations, and locations — mentioned across all text fields.
Requires `spacy` + `en_core_web_sm` model.

In [ ]:
if not SPACY_OK:
    print("Install spacy:  pip install spacy && python -m spacy download en_core_web_sm")
else:
    all_text_cols = [col_map.get(f) for f in
                     ["description","response","lessons_learned","lesson_title"]
                     if col_map.get(f)]
    all_texts = [t for col in all_text_cols for t in df[col].tolist()]

    ent_counter = Counter()
    for text in all_texts:
        doc = _nlp(str(text)[:500_000])
        for ent in doc.ents:
            if ent.label_ in {"ORG", "PERSON", "GPE", "PRODUCT", "EVENT"}:
                ent_counter[(ent.text.strip(), ent.label_)] += 1

    top_ents = ent_counter.most_common(TOP_N)

    if not top_ents:
        print("No named entities found in the text.")
    else:
        lbls = [f"{e} ({t})" for (e,t),_ in top_ents]
        vals = [cnt for _,cnt in top_ents]

        fig, ax = plt.subplots(figsize=(10, 5))
        hbar(ax, lbls, vals, C["accent3"], f"Top {TOP_N} Named Entities (NER)")
        plt.tight_layout(); plt.show()

        # Table
        rows = [{"Entity": e, "Type": t, "Mentions": cnt}
                for (e,t), cnt in top_ents]
        display(pd.DataFrame(rows))

---
## 1️⃣4️⃣ Word Co-occurrence Network

Words that frequently appear in the same record are connected by edges.
Node size = how connected the word is; edge thickness = co-occurrence strength.
Requires `networkx`.

In [ ]:
if not NX_OK:
    print("Install networkx:  pip install networkx")
else:
    all_text_cols = [col_map.get(f) for f in
                     ["description","response","lessons_learned","lesson_title"]
                     if col_map.get(f)]
    all_texts = [t for col in all_text_cols for t in df[col].tolist()]

    pair_counts = Counter()
    for text in all_texts:
        tokens = list(dict.fromkeys(tokenize(text, sw)))[:15]
        for i in range(len(tokens)):
            for j in range(i+1, len(tokens)):
                pair_counts[tuple(sorted([tokens[i], tokens[j]]))] += 1

    top_pairs = pair_counts.most_common(40)

    if len(top_pairs) < 5:
        print("Not enough text for a meaningful co-occurrence network.")
    else:
        G = nx.Graph()
        for (w1, w2), cnt in top_pairs:
            G.add_edge(w1, w2, weight=cnt)

        weights  = [G[u][v]["weight"] for u,v in G.edges()]
        max_w    = max(weights) if weights else 1
        n_sizes  = [300 + G.degree(n)*80 for n in G.nodes()]

        fig, ax = plt.subplots(figsize=(11, 9))
        fig.patch.set_facecolor("#FAFAFA"); ax.set_facecolor("#FAFAFA")
        pos = nx.spring_layout(G, seed=42, k=2.0)
        nx.draw_networkx_edges(G, pos, width=[w/max_w*3 for w in weights],
                               alpha=0.35, edge_color=C["accent2"], ax=ax)
        nx.draw_networkx_nodes(G, pos, node_size=n_sizes,
                               node_color=C["secondary"], alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=7, font_color="white",
                                font_weight="bold", ax=ax)
        ax.set_title("Word Co-occurrence Network", fontsize=13,
                     fontweight="bold", color=C["dark"], pad=10)
        ax.axis("off")
        plt.tight_layout(); plt.show()
        print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

---
## 1️⃣5️⃣ Sankey Flow Diagram

Shows how records flow from **Project Type → Stage → Category**.
An interactive Plotly chart — hover over flows to see exact counts.
Requires `plotly`.

In [ ]:
if not PLOTLY_OK:
    print("Install plotly:  pip install plotly")
else:
    cols = [col_map.get(f) for f in ["project_type","project_stage","new_category"]
            if col_map.get(f)]
    if len(cols) < 2:
        print("Need at least 2 of: project_type, project_stage, new_category.")
    else:
        all_labels, label_idx = [], {}
        for col in cols:
            for val in df[col].dropna().unique():
                key = f"{col}::{val}"
                if key not in label_idx:
                    label_idx[key] = len(all_labels)
                    all_labels.append(str(val))

        sources, targets, values = [], [], []
        for i in range(len(cols)-1):
            grp = df.groupby([cols[i], cols[i+1]]).size().reset_index(name="n")
            for _, row in grp.iterrows():
                s = label_idx.get(f"{cols[i]}::{row[cols[i]]}")
                t = label_idx.get(f"{cols[i+1]}::{row[cols[i+1]]}")
                if s is not None and t is not None:
                    sources.append(s); targets.append(t); values.append(int(row["n"]))

        field_labels = [c.replace("_"," ").title() for c in
                        ["Project Type","Stage","Category"][:len(cols)]]
        title = "Flow: " + " → ".join(field_labels)

        fig = go.Figure(go.Sankey(
            node=dict(label=all_labels, pad=15, thickness=20,
                      color="#0091D5"),
            link=dict(source=sources, target=targets, value=values,
                      color="rgba(0,145,213,0.25)"),
        ))
        fig.update_layout(title_text=title, font_size=12,
                          paper_bgcolor="#FAFAFA", height=500)
        fig.show()

---
## 1️⃣6️⃣ Semantic Similarity Heatmap

Uses sentence embeddings to measure *meaning* similarity between records — not just word overlap.
Records that score high (dark blue) are semantically similar and may be duplicates or related lessons.
Requires `sentence-transformers`.

In [ ]:
lc = col_map.get("lessons_learned") or col_map.get("description")
tc = col_map.get("lesson_title")

if not SBERT_OK or not lc:
    print("Install sentence-transformers:  pip install sentence-transformers")
elif len(df) < 3:
    print("Need at least 3 records.")
else:
    MAX_DOCS = 60
    texts  = [str(t) for t in df[lc].tolist()[:MAX_DOCS]]
    labels = (df[tc].tolist()[:MAX_DOCS] if tc
              else df[lc].apply(lambda t: str(t)[:28]).tolist()[:MAX_DOCS])

    print(f"Encoding {len(texts)} records with all-MiniLM-L6-v2 …")
    model      = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(texts, show_progress_bar=True)
    sim_matrix = sk_cosine(embeddings)

    size = max(8, len(texts) * 0.22)
    fig, ax = plt.subplots(figsize=(size, size * 0.9))
    im = ax.imshow(sim_matrix, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, shrink=0.8, label="Cosine Similarity")
    fs = max(5, 9 - len(texts)//10)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels([str(l)[:25] for l in labels], rotation=90, fontsize=fs)
    ax.set_yticklabels([str(l)[:25] for l in labels], fontsize=fs)
    ax.set_title("Semantic Similarity Matrix (Sentence Embeddings)",
                 fontsize=12, fontweight="bold", color=C["dark"], pad=10)
    plt.tight_layout(); plt.show()

    # Find most similar pairs
    import itertools
    pairs = []
    for i, j in itertools.combinations(range(len(texts)), 2):
        pairs.append((sim_matrix[i,j], str(labels[i])[:40], str(labels[j])[:40]))
    top_pairs_df = (pd.DataFrame(pairs, columns=["Similarity","Record A","Record B"])
                    .sort_values("Similarity", ascending=False).head(10))
    print("\nTop 10 most similar record pairs:")
    display(top_pairs_df.reset_index(drop=True).style.format({"Similarity": "{:.3f}"}))

---
## 📤 Export — Excel Workbook & PowerPoint

Generates the full **Excel workbook** (Raw Data + 10 insight sheets + embedded charts)
and **PowerPoint presentation** (all chart slides + key takeaways) using `analyze_project_data.py`.

> The `.py` script must be in the same folder as this notebook.

In [ ]:
import subprocess, sys

cmd = [sys.executable, "analyze_project_data.py", INPUT_FILE, "--out-dir", OUT_DIR]
if SHEET_NAME:
    cmd += ["--sheet", SHEET_NAME]

print("Running export pipeline …")
print("Command:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

---

## ✅ Summary

| Output | Location |
|--------|----------|
| Excel workbook | `{OUT_DIR}/{stem}_insights.xlsx` |
| PowerPoint deck | `{OUT_DIR}/{stem}_insights.pptx` |

**Tips:**
- Run cells top to bottom for a full analysis run.
- Skip any section by leaving its cell unrun — all sections are independent after Section 1.
- Change `TOP_N` in the Configuration cell to show more or fewer items in charts.
- The Sankey diagram (Section 15) is interactive — hover over flows in the notebook.
- The Semantic Similarity section (Section 16) shows the **top 10 most similar record pairs** below the heatmap — useful for finding duplicate or near-duplicate lessons.